# Lab · Segmentación de clientes
**Pregunta de negocio:** ¿a qué grupos de clientes les hablamos distinto, y cuánto vale cada grupo?

Este notebook replica en Python, paso por paso, el lab diseñado para Orange. Cada sección indica **qué widget de Orange reemplaza** y **por qué** se hace.

| Parte | Qué se hace | Equivalente Orange |
|---|---|---|
| 0 | Cargar datos | File |
| 1 | Conocer los datos | Data Table, Distributions |
| 2 | k-means | Select Columns, Normalize, k-Means, Scatter Plot, Group By |
| 3 | Comparar técnicas | Distances, Hierarchical Clustering, DBSCAN |
| 4 | Valor por segmento + guardar | Group By, Formula, Save Data |
| 5 | Cuando k-means falla | k-Means / DBSCAN sobre `formas_lunas.csv` |

> **Cómo usarlo:** ejecute las celdas en orden (`Shift + Enter`). Las celdas de texto con ✍️ son para que usted escriba sus respuestas.

## 0 · Preparar el entorno y cargar los archivos

**Qué hace la celda siguiente:**
1. Busca los CSV en la sesión de Colab.
2. Si no están, intenta clonar su repositorio de GitHub (funciona si el repo es **público**).
3. Si tampoco, le abre un cuadro para **subir los archivos** desde su computadora.

> Importante: abrir un notebook desde GitHub en Colab **no** trae los demás archivos del repo. Colab arranca en una máquina vacía; por eso hay que clonar o subir los datos.

👉 Si sus CSV están en otra carpeta del repo, cambie `CARPETA`.

In [ ]:
import os, subprocess

REPO    = "chartie87742/marketing-analytics-carlos-garciagranados"
CARPETA = "lab_segmentacion"          # carpeta del repo donde están el notebook y /data
ARCHIVOS = ["clientes_retail.csv", "formas_lunas.csv"]

def buscar(nombre):
    candidatos = [nombre, f"data/{nombre}", f"{CARPETA}/data/{nombre}",
                  f"/content/repo/{CARPETA}/data/{nombre}", f"/content/repo/data/{nombre}",
                  f"/content/repo/{nombre}"]
    return next((c for c in candidatos if os.path.exists(c)), None)

faltan = [a for a in ARCHIVOS if buscar(a) is None]

if faltan and not os.path.exists("/content/repo"):
    os.environ["GIT_TERMINAL_PROMPT"] = "0"   # si el repo es privado, falla rápido en vez de quedarse esperando
    r = subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", "/content/repo"],
                       capture_output=True, text=True)
    print("Clonado del repo:", "OK" if r.returncode == 0 else "no se pudo (¿repo privado?)")
    faltan = [a for a in ARCHIVOS if buscar(a) is None]

if faltan:
    try:
        from google.colab import files
        print("Suba estos archivos:", faltan)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"No encuentro {faltan}. Póngalos en la carpeta data/.")

RUTA = {a: buscar(a) for a in ARCHIVOS}
RUTA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import joblib

SEMILLA = 42                     # fija el azar para que k-means dé siempre el mismo resultado
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.dpi"] = 110
COLORES = {"C1": "#4C78A8", "C2": "#F58518", "C3": "#54A24B"}

## Parte 1 · Conozca los datos (≈10 min)
**Orange:** `File → Data Table` y `Distributions (gasto_total)`.

Primero miramos la tabla: cuántas filas y columnas, de qué tipo es cada columna y si hay vacíos.

In [ ]:
clientes = pd.read_csv(RUTA["clientes_retail.csv"])
print("Filas x columnas:", clientes.shape)
display(clientes.head())
clientes.info()

In [ ]:
display(clientes.describe().T)   # media, desviación, mínimo, percentiles y máximo de cada columna

### Distribución de `gasto_total`
Equivale a `Distributions`. Además del histograma calculamos tres cosas que Orange no muestra directamente:
- **Media vs. mediana:** si la media es bastante mayor que la mediana, unos pocos clientes grandes jalan el promedio hacia arriba.
- **Asimetría (skew):** > 1 indica cola larga a la derecha.
- **Concentración:** qué % del gasto aporta el 20 % de clientes que más gasta (la lógica de Pareto).

In [ ]:
g = clientes["gasto_total"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(g, bins=40, color="#4C78A8", edgecolor="white")
ax[0].axvline(g.mean(),   color="red",   ls="--", label=f"Media {g.mean():,.0f}")
ax[0].axvline(g.median(), color="black", ls=":",  label=f"Mediana {g.median():,.0f}")
ax[0].set(title="Distribución de gasto_total", xlabel="gasto_total", ylabel="Nº de clientes"); ax[0].legend()
ax[1].boxplot(g); ax[1].set(title="Boxplot de gasto_total", ylabel="gasto_total", xticks=[])
plt.tight_layout(); plt.show()

top20 = g.sort_values(ascending=False).head(int(len(g) * 0.20)).sum() / g.sum()
print(f"Media: {g.mean():,.0f} | Mediana: {g.median():,.0f} | Asimetría: {g.skew():.2f}")
print(f"El 20% de clientes que más gasta genera el {top20:.0%} del gasto total")

✍️ **Respuesta Parte 1 — ¿Cómo se ve `gasto_total`?**

_Escriba aquí, usando los números de arriba. Guía:_ distribución sesgada a la derecha (media > mediana); muchos clientes gastan poco y pocos gastan mucho; el top 20 % concentra X % del gasto.

> **Implicación técnica (para tenerla presente):** k-means usa promedios, y los promedios se dejan jalar por los extremos. Con una variable así de sesgada, los clientes "grandes" tienden a definir su propio grupo. El lab no lo pide, pero una mejora habitual es aplicar logaritmo a `gasto_total` e `ingreso_anual` antes de agrupar.

## Parte 2 · k-means: el punto de partida (≈15 min)

### 2.1 Seleccionar atributos — `Select Columns`
Solo entran al algoritmo las 9 variables que pide la guía. Las demás (gasto por categoría, `gasto_premium`, respuesta a campaña…) quedan fuera del cálculo pero **siguen en la tabla** para describir los grupos después (eso es lo que Orange llama *Meta*).

¿Por qué dejar fuera `respondio_ultima_campana`? Porque es lo que queremos **explicar** con los segmentos; si la metemos en el agrupamiento, el resultado se vuelve circular.

In [ ]:
ATRIBUTOS = ["ingreso_anual", "gasto_total", "compras_tienda", "compras_web", "compras_catalogo",
             "visitas_web_mes", "recencia_dias", "hijos_pequenos", "edad"]

faltantes = [c for c in ATRIBUTOS if c not in clientes.columns]
assert not faltantes, f"Revise nombres de columnas: {faltantes}"

X = clientes[ATRIBUTOS]
print("Valores vacíos en los atributos:", int(X.isna().sum().sum()))

### 2.2 Estandarizar — `Normalize` (Standardize μ=0, σ²=1)
k-means mide **distancias**. Si no estandarizamos, una diferencia de 10,000 en ingreso pesa muchísimo más que una diferencia de 5 visitas web, y el algoritmo prácticamente solo "ve" ingreso y gasto.

Estandarizar = a cada valor se le resta la media de su columna y se divide entre la desviación estándar (z-score). Así todas las variables quedan en la misma escala: 0 = promedio, +1 = una desviación por encima.

In [ ]:
escalador = StandardScaler()
Xz = escalador.fit_transform(X)

# Verificación: todas las columnas deben tener media 0 y desviación 1
display(pd.DataFrame(Xz, columns=ATRIBUTOS).agg(["mean", "std"]).round(2))

### 2.3 ¿Cuántos grupos? — `k-Means` "From 2 to 8"
Probamos k = 2…8 y para cada uno calculamos:
- **Silueta (silhouette):** para cada cliente compara qué tan cerca está de su propio grupo vs. del grupo vecino. Va de −1 a 1. Referencia común: > 0.50 estructura fuerte; 0.25–0.50 moderada/débil; < 0.25 casi no hay grupos naturales.
- **Inercia (método del codo):** suma de distancias al centro del grupo. Siempre baja al subir k; buscamos el "codo".

`n_init=10` = corre k-means 10 veces con centros iniciales distintos y se queda con el mejor (Orange también reinicia varias veces).

In [ ]:
filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Xz)
    filas.append({"k": k, "silueta": silhouette_score(Xz, km.labels_), "inercia": km.inertia_})
tabla_k = pd.DataFrame(filas).set_index("k")
display(tabla_k.style.highlight_max(subset="silueta", color="#d4edda").format({"silueta": "{:.3f}", "inercia": "{:,.0f}"}))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
tabla_k["silueta"].plot(ax=ax[0], marker="o", title="Silueta por k (más alto = mejor)")
tabla_k["inercia"].plot(ax=ax[1], marker="o", color="gray", title="Inercia por k (buscar el codo)")
for a in ax: a.set_xlabel("k"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"Mejor silueta: k = {tabla_k['silueta'].idxmax()} ({tabla_k['silueta'].max():.3f})")

### 2.4 Fijar k = 3 y nombrar los grupos C1, C2, C3
k-means numera los grupos al azar (0, 1, 2). Para que el resultado sea **estable y legible**, los reordenamos por gasto promedio: **C1 = menor gasto, C3 = mayor gasto**.

> Sus etiquetas en Orange pueden quedar en otro orden. Lo que importa es el **perfil** de cada grupo, no el número.

In [ ]:
K = 3
km3 = KMeans(n_clusters=K, n_init=10, random_state=SEMILLA).fit(Xz)

orden = clientes.groupby(km3.labels_)["gasto_total"].mean().sort_values().index
MAPA = {original: f"C{i+1}" for i, original in enumerate(orden)}
clientes["Cluster"] = pd.Series(km3.labels_).map(MAPA).values

print(f"Silueta con k=3: {silhouette_score(Xz, km3.labels_):.3f}")
clientes["Cluster"].value_counts().sort_index()

### 2.5 Scatter Plot — x = ingreso_anual, y = gasto_total, color = Cluster

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for c in sorted(clientes["Cluster"].unique()):
    s = clientes[clientes["Cluster"] == c]
    ax.scatter(s["ingreso_anual"], s["gasto_total"], s=12, alpha=.6, label=f"{c} (n={len(s)})", color=COLORES[c])
ax.set(xlabel="ingreso_anual", ylabel="gasto_total", title="Segmentos k-means (k=3)")
ax.legend(); ax.grid(alpha=.3); plt.show()

### 2.6 Perfil de cada segmento — `Group By` (promedio por Cluster)
La primera tabla muestra el **promedio real** de cada variable. La segunda es un **índice vs. el promedio de todos los clientes** (100 = igual al promedio; 150 = 50 % arriba; 60 = 40 % abajo). El índice es lo que más ayuda a ponerle nombre de negocio a cada grupo.

In [ ]:
EXTRA = [c for c in ["gasto_premium", "respondio_ultima_campana"] if c in clientes.columns]
if "respondio_ultima_campana" in clientes and clientes["respondio_ultima_campana"].dtype == object:
    clientes["respondio_ultima_campana"] = (clientes["respondio_ultima_campana"].astype(str).str.strip().str.lower()
                                            .isin(["1", "si", "sí", "yes", "true"]).astype(int))

perfil = clientes.groupby("Cluster")[ATRIBUTOS + EXTRA].mean()
perfil.insert(0, "n_clientes", clientes["Cluster"].value_counts().sort_index())
display(perfil.T)

indice = (clientes.groupby("Cluster")[ATRIBUTOS].mean() / clientes[ATRIBUTOS].mean() * 100).round(0)
display(indice.T.style.background_gradient(cmap="RdYlGn", axis=1).format("{:.0f}"))

In [ ]:
# Equivalente a Box Plot por Cluster: la mediana y la dispersión de cada variable por segmento
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), ATRIBUTOS):
    datos = [clientes.loc[clientes["Cluster"] == c, col] for c in sorted(clientes["Cluster"].unique())]
    ax.boxplot(datos, showfliers=False)
    ax.set_xticks(range(1, len(datos) + 1), sorted(clientes["Cluster"].unique()))
    ax.set_title(col, fontsize=10)
plt.suptitle("Variables por segmento (sin outliers)"); plt.tight_layout(); plt.show()

✍️ **Respuestas Parte 2**
1. **¿Qué k tiene mejor silueta?** _k = …, silueta = …_
2. **¿Es muy alta?** _Compare con la referencia de 2.3. Si queda entre 0.2 y 0.3 (lo típico en datos de clientes reales), la respuesta honesta es: no; los grupos se traslapan y son una partición útil para negocio más que "grupos naturales"._
3. **Nombre de negocio por segmento** _(use la tabla de índices; nombre = quién es + cómo compra)_:
   - C1: …
   - C2: …
   - C3: …

> **Hipótesis a verificar con sus números** (en esta base suele salir algo parecido, pero confírmelo): un grupo de ingreso y gasto altos, sin hijos pequeños, que compra en tienda y catálogo (*"Premium / alto valor"*); uno de ingreso bajo, con hijos pequeños, muchas visitas web y poco gasto (*"Familias sensibles al precio"*); y uno intermedio, de más edad, que compra mucho por web y tienda (*"Establecidos / compradores de valor"*).

## Parte 3 · Compare técnicas (≈15 min)

### 3.1 Jerárquico — `Distances (Euclidean) → Hierarchical Clustering (Ward)`, cortar en 3
**Cómo funciona:** empieza con cada cliente como su propio grupo y va uniendo los dos grupos más parecidos hasta que queda uno solo. **Ward** une el par que menos aumenta la varianza interna (por eso tiende a dar grupos parecidos a k-means). El **dendrograma** muestra esas uniones; cortar en 3 = trazar una línea horizontal que cruce 3 ramas.

In [ ]:
Z = linkage(Xz, method="ward")        # Ward usa distancia euclidiana internamente

plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=Z[-(K - 1), 2], no_labels=True)
plt.axhline(Z[-(K - 1), 2], color="red", ls="--", label="Corte en 3 grupos")
plt.title("Dendrograma (Ward) — últimas 30 uniones"); plt.ylabel("Distancia de unión"); plt.legend(); plt.show()

ward = fcluster(Z, t=K, criterion="maxclust")
orden_w = clientes.groupby(ward)["gasto_total"].mean().sort_values().index
clientes["Cluster_ward"] = pd.Series(ward).map({o: f"H{i+1}" for i, o in enumerate(orden_w)}).values
print(f"Silueta Ward (3 grupos): {silhouette_score(Xz, ward):.3f}")
clientes["Cluster_ward"].value_counts().sort_index()

### 3.2 DBSCAN — elegir `eps` con la distancia al k-ésimo vecino
**Cómo funciona:** un cliente es "núcleo" si tiene al menos `min_samples` vecinos a distancia ≤ `eps`. Los núcleos conectados forman un grupo; lo que no alcanza a ningún grupo es **ruido (−1)**. No hay que decirle cuántos grupos quiere.

**Gráfico de k-distancia:** para cada cliente se mide la distancia a su k-ésimo vecino más cercano y se ordenan de menor a mayor. Donde la curva "se dispara" (el codo) es un buen `eps`: por debajo, los puntos están en zonas densas; por encima, son aislados.

`MIN_VECINOS = 5` (igual que en la Parte 5). Con 9 variables, una regla común es usar más (≈ 2 × nº de variables = 18); puede probarlo después.

In [ ]:
MIN_VECINOS = 5
dist, _ = NearestNeighbors(n_neighbors=MIN_VECINOS).fit(Xz).kneighbors(Xz)
kdist = np.sort(dist[:, -1])

plt.figure(figsize=(9, 4))
plt.plot(kdist)
for e, col in [(1.5, "green"), (2, "orange"), (3, "red")]:
    plt.axhline(e, ls="--", color=col, label=f"eps = {e}")
plt.title(f"Distancia al {MIN_VECINOS}º vecino (ordenada)"); plt.xlabel("Clientes ordenados"); plt.ylabel("Distancia")
plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
def resumen_dbscan(eps, datos=Xz, min_s=MIN_VECINOS):
    lab = DBSCAN(eps=eps, min_samples=min_s).fit_predict(datos)
    grupos = sorted(set(lab) - {-1})
    tam = pd.Series(lab).value_counts()
    return lab, {"eps": eps, "n_grupos": len(grupos), "ruido": int((lab == -1).sum()),
                 "% ruido": (lab == -1).mean(), "tamaños": [int(tam[g]) for g in grupos]}

filas, etiquetas_db = [], {}
for eps in [1.0, 1.5, 2.0, 2.5, 3.0]:
    lab, fila = resumen_dbscan(eps)
    etiquetas_db[eps] = lab; filas.append(fila)
display(pd.DataFrame(filas).style.format({"% ruido": "{:.1%}"}))

EPS_ELEGIDO = 1.5
lab = etiquetas_db[EPS_ELEGIDO]
clientes["Cluster_dbscan"] = np.where(lab == -1, "Ruido", "D" + (lab + 1).astype(str))

### 3.3 Comparar cuántos clientes caen en cada grupo — `Merge Data` / Scatter coloreado
- **Tamaños por técnica**.
- **Tabla cruzada k-means vs. Ward:** si las técnicas coinciden, cada fila tiene casi todo en una sola columna.
- **ARI (Adjusted Rand Index):** 1 = particiones idénticas, 0 = coincidencia al azar.

In [ ]:
tamanos = pd.concat({
    "k-means": clientes["Cluster"].value_counts().sort_index(),
    "Ward": clientes["Cluster_ward"].value_counts().sort_index(),
    f"DBSCAN eps={EPS_ELEGIDO}": clientes["Cluster_dbscan"].value_counts().sort_index(),
}, axis=1)
display(tamanos)

display(pd.crosstab(clientes["Cluster"], clientes["Cluster_ward"], margins=True, margins_name="Total"))
print(f"ARI k-means vs Ward:   {adjusted_rand_score(clientes['Cluster'], clientes['Cluster_ward']):.2f}")
print(f"ARI k-means vs DBSCAN: {adjusted_rand_score(clientes['Cluster'], clientes['Cluster_dbscan']):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, col, titulo in zip(axes, ["Cluster", "Cluster_ward", "Cluster_dbscan"],
                           ["k-means (k=3)", "Jerárquico Ward (3)", f"DBSCAN (eps={EPS_ELEGIDO})"]):
    for nombre in sorted(clientes[col].unique()):
        s = clientes[clientes[col] == nombre]
        ax.scatter(s["ingreso_anual"], s["gasto_total"], s=8, alpha=.6, label=f"{nombre} ({len(s)})",
                   color="lightgray" if nombre == "Ruido" else None)
    ax.set(title=titulo, xlabel="ingreso_anual"); ax.legend(fontsize=8, markerscale=2)
axes[0].set_ylabel("gasto_total"); plt.tight_layout(); plt.show()

✍️ **Respuestas Parte 3**
1. **¿Las tres técnicas dan los mismos grupos?** _Use el ARI y la tabla cruzada. Lo esperable: k-means y Ward se parecen bastante (ambos buscan grupos compactos); DBSCAN da algo muy distinto._
2. **¿Qué pasa con DBSCAN al subir eps?** _Vea la tabla de eps: al subir eps los vecindarios crecen, el ruido baja y los grupos se fusionan hasta quedar prácticamente uno solo. Esto indica que los clientes forman una nube continua, sin huecos de baja densidad que los separen._
3. **¿Cuál le serviría más a marketing para actuar?** _Argumente con: todos los clientes asignados (sin "ruido" que nadie atiende), grupos de tamaño razonable, perfiles interpretables, y poder clasificar clientes nuevos (k-means sí puede; Ward y DBSCAN no directamente)._

## Parte 4 · ¿Cuánto vale cada segmento? (≈10 min)
**Orange:** `Group By` (conteo, promedio, suma) + `Formula` (porcentajes).

Calculamos por segmento: nº de clientes, gasto promedio, gasto total, **% de clientes**, **% del gasto**, y la **tasa de respuesta** a la última campaña.

Dos columnas extra que ayudan a decidir:
- **Índice de valor = % gasto ÷ % clientes.** > 1 significa que el segmento aporta más de lo que pesa en número.
- **Valor esperado por contacto (proxy) = tasa de respuesta × gasto promedio.** _Supuesto:_ usa el gasto histórico como aproximación del valor de un respondedor; no es margen ni CLV real.

In [ ]:
valor = clientes.groupby("Cluster").agg(
    clientes=("gasto_total", "size"),
    gasto_promedio=("gasto_total", "mean"),
    gasto_total=("gasto_total", "sum"),
    ingreso_promedio=("ingreso_anual", "mean"),
    tasa_respuesta=("respondio_ultima_campana", "mean"),
    respondieron=("respondio_ultima_campana", "sum"),
)
valor["% clientes"]       = valor["clientes"] / valor["clientes"].sum()
valor["% gasto"]          = valor["gasto_total"] / valor["gasto_total"].sum()
valor["% respondedores"]  = valor["respondieron"] / valor["respondieron"].sum()
valor["indice_valor"]     = valor["% gasto"] / valor["% clientes"]
valor["valor_esp_contacto"] = valor["tasa_respuesta"] * valor["gasto_promedio"]

display(valor.style.format({
    "gasto_promedio": "{:,.0f}", "gasto_total": "{:,.0f}", "ingreso_promedio": "{:,.0f}",
    "tasa_respuesta": "{:.1%}", "respondieron": "{:,.0f}", "% clientes": "{:.1%}", "% gasto": "{:.1%}",
    "% respondedores": "{:.1%}", "indice_valor": "{:.2f}", "valor_esp_contacto": "{:,.0f}"}))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
valor[["% clientes", "% gasto", "% respondedores"]].mul(100).plot.bar(ax=ax[0], rot=0)
ax[0].set(title="Peso de cada segmento", ylabel="%"); ax[0].grid(axis="y", alpha=.3)
valor["tasa_respuesta"].mul(100).plot.bar(ax=ax[1], rot=0, color=[COLORES[c] for c in valor.index])
ax[1].axhline(clientes["respondio_ultima_campana"].mean() * 100, ls="--", color="black", label="Promedio general")
ax[1].set(title="Tasa de respuesta a la última campaña", ylabel="%"); ax[1].legend()
plt.tight_layout(); plt.show()

✍️ **Respuestas Parte 4**
1. **¿Qué % de clientes genera qué % del gasto?** _Ej.: "C3 es el X % de clientes y genera el Y % del gasto (índice Z)"._
2. **¿A qué segmento le gastaría el presupuesto de la próxima campaña y por qué?** Conecte con CLV/CAC:
   - CLV aproximado ∝ gasto promedio × probabilidad de que siga comprando (recencia baja = más activo).
   - CAC/costo por contacto similar entre segmentos → gane el que tenga mayor **valor esperado por contacto**.
   - _Advertencia crítica:_ una tasa de respuesta alta **no prueba que la campaña cause** la compra; esos clientes quizá habrían comprado igual. Lo correcto es probar con un grupo de control (A/B) antes de asignar todo el presupuesto.
   - Considere también el tamaño: un segmento muy rentable pero pequeño puede agotarse rápido; el segundo mejor quizá sea donde está el crecimiento.

### Guardar la segmentación — `Save Data` → `clientes_con_cluster.csv`
Se guarda la tabla original **+ la columna `Cluster`** (C1, C2, C3), igual que la salida de Orange. Las columnas de Ward y DBSCAN eran solo para comparar, no se incluyen.

Además se guarda el modelo (estandarizador + k-means) en `.joblib`; sirve para clasificar clientes nuevos y lo usa la app de Streamlit.

In [ ]:
os.makedirs("outputs", exist_ok=True)
salida = clientes.drop(columns=["Cluster_ward", "Cluster_dbscan"])
salida.to_csv("outputs/clientes_con_cluster.csv", index=False)

# Comprobación (lo que pide la guía: abrirlo y ver la columna Cluster con C1, C2, C3)
check = pd.read_csv("outputs/clientes_con_cluster.csv")
assert "Cluster" in check.columns and set(check["Cluster"]) == {"C1", "C2", "C3"}
print("✔ Archivo correcto:", check.shape)
display(check["Cluster"].value_counts().sort_index())

modelo = Pipeline([("estandarizar", escalador), ("kmeans", km3)])
joblib.dump({"modelo": modelo, "atributos": ATRIBUTOS, "mapa_etiquetas": MAPA}, "outputs/modelo_kmeans_k3.joblib")
print("✔ Modelo guardado: outputs/modelo_kmeans_k3.joblib")

In [ ]:
# Solo en Colab: descargar los archivos a su computadora
try:
    from google.colab import files
    files.download("outputs/clientes_con_cluster.csv")
    files.download("outputs/modelo_kmeans_k3.joblib")
except ImportError:
    print("No está en Colab: los archivos quedaron en la carpeta outputs/")

## Parte 5 · Datos sintéticos: cuando k-means falla (≈10 min)
`formas_lunas.csv`: dos lunas entrelazadas + 20 puntos de ruido. Sabemos la respuesta correcta (`grupo_real`), así que podemos medir qué técnica acierta.

**No se estandariza**: x e y ya están en la misma escala, y los valores de eps de la guía (0.2 y 0.1) están pensados para las unidades originales.

In [ ]:
lunas = pd.read_csv(RUTA["formas_lunas.csv"])
print(lunas.shape); display(lunas.head())
display(lunas["grupo_real"].value_counts())
XY = lunas[["x", "y"]].values

In [ ]:
pred = {"Real (grupo_real)": lunas["grupo_real"].astype(str).values}
pred["k-means k=2"] = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit_predict(XY).astype(str)
for eps in [0.2, 0.1]:
    pred[f"DBSCAN eps={eps}"] = DBSCAN(eps=eps, min_samples=5).fit_predict(XY).astype(str)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), sharex=True, sharey=True)
for ax, (titulo, lab) in zip(axes, pred.items()):
    for v in sorted(set(lab)):
        m = lab == v
        es_ruido = v in ("-1", "ruido", "Ruido", "noise")
        ax.scatter(XY[m, 0], XY[m, 1], s=10, alpha=.8, label=("Ruido" if es_ruido else v) + f" ({m.sum()})",
                   color="black" if es_ruido else None, marker="x" if es_ruido else "o")
    ax.set_title(titulo); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

real = pred["Real (grupo_real)"]
for titulo, lab in list(pred.items())[1:]:
    print(f"{titulo:18s} ARI vs real = {adjusted_rand_score(real, lab):.2f} | grupos = {len(set(lab) - {'-1'})} | "
          f"ruido = {(lab == '-1').sum()}")

✍️ **Respuestas Parte 5**
1. **¿Por qué k-means corta mal?** _k-means asigna cada punto al centro más cercano, así que siempre corta el espacio con una línea recta entre centros y supone grupos redondos ("bolas") de tamaño parecido. Una luna es curva: la punta de una queda más cerca del centro de la otra._
2. **¿Qué hizo DBSCAN con el ruido?** _Con eps=0.2 sigue la forma de las lunas por densidad y deja los puntos aislados como ruido (−1). Con eps=0.1 el vecindario es demasiado pequeño: parte las lunas en pedazos y marca mucho más ruido. Moraleja: DBSCAN es muy sensible a eps._
3. **¿Cuándo sospecharía que sus clientes no tienen forma de "bolas"?** _Silueta baja con cualquier k; scatter con formas alargadas, curvas o de densidad muy distinta; k-means da resultados diferentes con otra semilla; o variables muy sesgadas (como gasto_total). En esos casos: transformar (log), probar mezclas gaussianas (grupos elípticos — ver el notebook de la tarea) o DBSCAN/HDBSCAN._

## Cierre — recomendación a marketing (3 líneas)
✍️ _Plantilla (ajústela con sus números):_

1. Recomiendo segmentar con **k-means, k = 3**, sobre las 9 variables estandarizadas: asigna a todos los clientes, da perfiles claros y permite clasificar clientes nuevos.
2. El segmento **[nombre]** concentra **[X] % del gasto con [Y] % de clientes** y la mayor tasa de respuesta, así que recibe la mayor parte del presupuesto; **[nombre]** se atiende con mensajes de precio y canal web.
3. La silueta (~[valor]) indica grupos que se traslapan: tratarlos como una **partición de negocio**, no como grupos naturales, y validar la campaña con un grupo de control.